# GPUPerf-3B: Profiler-Grounded GPU Optimization Model
## 🚀 Quad-GPU Training & Live Hardware-Conditional Inference

This turnkey notebook trains and evaluates **GPUPerf-3B** (`Qwen2.5-Coder-3B-Instruct`) using **100% real physical GPU measurements** harvested across 4 distinct NVIDIA GPU generations:
- 🟦 **Turing (Tesla T4, `sm_75`)**: 40 SMs, 320 GB/s GDDR6
- 🟩 **Ampere (NVIDIA A100-SXM4-80GB, `sm_80`)**: 108 SMs, 2,039 GB/s HBM2e
- 🟧 **Ada Lovelace (NVIDIA L4, `sm_89`)**: 58 SMs, 300 GB/s GDDR6
- 🟪 **Blackwell (RTX PRO 6000 Blackwell Server Edition, `sm_120`)**: 188 SMs, 500 GB/s

### Total Ground Truth Scale:
- **490,209 physical kernel runs** under steady-state thermal conditions.
- **1,577 unique configurations** aggregated into **949 Full-Code SFT winners** and **949 DPO preference pairs**.
- Fast fine-tuning powered by **Unsloth 4-bit QLoRA** (runs in 10-15 minutes on free/Pro Colab).

In [ ]:
# 1. Verify Attached Hardware
!nvidia-smi

In [ ]:
# 2. Fast Install Unsloth & Modern Fine-Tuning Stack
%%capture
!pip install --no-deps "xformers<0.0.29" triton
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets

In [ ]:
# 3. Clone Repository & Setup Dataset
import os
if not os.path.exists("GPUPerf"):;
    !git clone https://github.com/sarimahsan/GPUPerf.git
%cd GPUPerf
!git pull
!ls -lh ./data

In [ ]:
# 4. Load Base Model: Qwen2.5-Coder-3B-Instruct (4-bit QLoRA via Unsloth)
import torch
from unsloth import FastLanguageModel

max_seq_length = 3072
model_name = "Qwen/Qwen2.5-Coder-3B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print("Model loaded and LoRA adapters successfully initialized!")

In [ ]:
# 5. Load & Format Real Quad-GPU SFT Dataset
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",
)

raw_dataset = load_dataset("json", data_files="./data/sft_multi_gpu.jsonl", split="train")
print(f"Loaded {len(raw_dataset)} verified physical training samples across 4 GPU architectures!")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

dataset = raw_dataset.map(formatting_prompts_func, batched=True)
print("Sample tokenized chat prompt:")
print(dataset[0]["text"][:600] + "...")

In [ ]:
# 6. Train Stage 1: Supervised Fine-Tuning (SFT)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="./checkpoints/gpuperf_sft_model",
        report_to="none"
    ),
)

print("Starting Quad-GPU SFT training...")
trainer_stats = trainer.train()
print("SFT Training Finished Successfully!")

In [ ]:
# 7. Save Fine-Tuned Adapters
output_model_dir = "./checkpoints/gpuperf_3b_pilot"
model.save_pretrained(output_model_dir)
tokenizer.save_pretrained(output_model_dir)
print(f"Saved GPUPerf-3B adapter to {output_model_dir}")

In [ ]:
# 8. Live Interactive Hardware-Conditional Inference
# Enable ultra-fast Unsloth native inference
FastLanguageModel.for_inference(model)

def optimize_for_gpu(device_name, arch_tag, sms, bw_gbs, latency_us, initial_tile_desc):
    prompt_text = f"""Here is a TRITON GPU kernel and its hardware profiling report from NVIDIA Nsight Compute:

```triton
{initial_tile_desc}
```

[PROFILER REPORT]
Device: {device_name} ({arch_tag}, {sms} SMs, {bw_gbs} GB/s DRAM)
Kernel: gemm_kernel
Measured Latency: {latency_us:.1f} us
Occupancy: Achieved 42.0%
Bottleneck: ResourceContention
[/PROFILER REPORT]

Analyze the profiler metrics and write an optimized kernel for {device_name}."""

    messages = [
        {"role": "system", "content": "You are GPUPerf, a specialized GPU systems optimization engineer. Analyze GPU kernels and profiler evidence, diagnose root-cause bottlenecks, and provide an optimized, empirically faster kernel."},
        {"role": "user", "content": prompt_text}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(input_ids=inputs, max_new_tokens=768, temperature=0.15)
    generated_text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    
    print(f"\n{'='*70}")
    print(f"OPTIMIZATION RESULT FOR: {device_name} ({arch_tag})")
    print(f"{'='*70}")
    print(generated_text)

# Test Baseline Code snippet
baseline_gemm = """# Baseline Triton GEMM
BLOCK_M: tl.constexpr = 64,
BLOCK_N: tl.constexpr = 64,
BLOCK_K: tl.constexpr = 32,
# Launch Config: num_warps=4, num_stages=2"""

# 1. Test on Tesla T4 (Turing, sm_75) -> Should favor conservative shared-memory tiles
optimize_for_gpu("Tesla T4", "sm_75", 40, 320.0, 5235.2, baseline_gemm)

# 2. Test on NVIDIA A100 (Ampere, sm_80) -> Should favor large reduction tiles BLOCK_K=64 to leverage 2039 GB/s HBM2e
optimize_for_gpu("NVIDIA A100-SXM4-80GB", "sm_80", 108, 2039.0, 93.3, baseline_gemm)

# 3. Test on NVIDIA RTX PRO 6000 Blackwell (Blackwell, sm_120) -> Should exploit 188 SMs with high stages
optimize_for_gpu("NVIDIA RTX PRO 6000 Blackwell Server Edition", "sm_120", 188, 500.0, 59.6, baseline_gemm)

## 🎯 (Optional) Stage 2: Direct Preference Optimization (DPO)
Align the model's preference against catastrophic hardware pitfalls using the empirical DPO dataset.

In [ ]:
# 9. Optional DPO Alignment Step
from trl import DPOTrainer, DPOConfig
from datasets import load_dataset
from unsloth import PatchDPOTrainer
PatchDPOTrainer()

dpo_raw = load_dataset("json", data_files="./data/dpo_multi_gpu.jsonl", split="train")
print(f"Loaded {len(dpo_raw)} empirical DPO preference pairs!")

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    tokenizer=tokenizer,
    beta=0.1,
    train_dataset=dpo_raw,
    max_length=max_seq_length,
    max_prompt_length=1536,
    args=DPOConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_ratio=0.1,
        num_train_epochs=1,
        learning_rate=5e-6,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        seed=42,
        output_dir="./checkpoints/gpuperf_dpo_model",
        report_to="none"
    ),
)

# Uncomment to run DPO training:
# dpo_trainer.train()
# model.save_pretrained("./checkpoints/gpuperf_3b_dpo_final")